# 🌸 Iris Flower Classification with K-Nearest Neighbors
**DecodeLabs AI Internship — Project 2**

**Pipeline:** Iris Dataset → StandardScaler → KNN → Confusion Matrix + F1 Score  
**Goal:** Demonstrate mastery of the full supervised learning lifecycle — not just accuracy numbers, but *understanding why each step matters.*

---
### IPO Framework
| Stage | Components |
|-------|-----------|
| **Input** | Iris dataset (150 samples, 4 features, 3 classes) + StandardScaler |
| **Process** | Train/Test Split → KNN with elbow-tuned K |
| **Output** | Confusion Matrix, F1 Score, Algorithm Comparison |


## 1. Environment Setup & Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix, classification_report,
    f1_score, accuracy_score
)

# Reproducibility — every run produces identical results
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
plt.style.use('seaborn-v0_8-whitegrid')

print("All libraries loaded successfully.")
print(f"NumPy {np.__version__} | Pandas {pd.__version__}")


## 2. Data Loading & Exploration

The Iris dataset is the *canonical* benchmark for classification. Three species of iris flowers
are described by four physical measurements:
- **Sepal length / width** — the outer protective leaf
- **Petal length / width** — the inner flowering part

With 50 samples per class, it is perfectly balanced — no class imbalance headaches here.


In [ ]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target, name='species')
target_names = iris.target_names

print(f"Dataset dimensions : {X.shape}  ({X.shape[0]} samples × {X.shape[1]} features)")
print(f"Classes            : {list(target_names)}")
print(f"Samples per class  :\n{y.value_counts().sort_index().rename(dict(enumerate(target_names)))}\n")
print("First 5 rows:")
X.head()


In [ ]:
print("Descriptive Statistics:")
X.describe().round(3)


In [ ]:
# Visualise each feature's distribution split by species
colors = ['#E74C3C', '#3498DB', '#2ECC71']
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, feature in enumerate(iris.feature_names):
    ax = axes[idx // 2][idx % 2]
    for class_id, color in zip(range(3), colors):
        mask = (y == class_id)
        ax.hist(X.loc[mask, feature], bins=15, alpha=0.65,
                color=color, label=target_names[class_id], edgecolor='white')
    label = feature.replace(' (cm)', '').title()
    ax.set_title(label, fontsize=11, fontweight='bold')
    ax.set_xlabel('Value (cm)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)

plt.suptitle('Feature Distributions by Iris Species', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Observation: Petal dimensions are the most discriminative features.")


## 3. Feature Scaling with StandardScaler

**Why this matters for KNN:**  
KNN classifies by measuring *distance* in feature space. If sepal length (range 4–8 cm) 
and petal width (range 0.1–2.5 cm) are fed raw, the larger-scale feature will dominate 
every distance calculation — even if petal width is more informative.

`StandardScaler` transforms each feature:  
$$z = \frac{x - \mu}{\sigma}$$

After scaling: **mean = 0, variance = 1** for every feature. All four dimensions 
now compete on equal footing.

> ⚠️ **Critical detail:** `.fit_transform()` is called on training data only. 
> The test set is `.transform()`-ed using the *training* mean/std — never fitted on test data, 
> as that would leak future information.


In [ ]:
scaler = StandardScaler()

# Fit on all X here for demo; in the pipeline below we fit only on X_train
X_scaled_demo = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled_demo, columns=X.columns)

comparison = pd.concat([
    X.describe().loc[['mean', 'std']].rename(index={'mean': 'mean (raw)', 'std': 'std (raw)'}),
    X_scaled_df.describe().loc[['mean', 'std']].rename(index={'mean': 'mean (scaled)', 'std': 'std (scaled)'})
])
print("Before vs After StandardScaler:")
comparison.round(4)


## 4. Train / Test Split

**80/20 split** — industry standard for datasets of this size.  
- Training set (120 samples) teaches the model the decision boundaries.  
- Test set (30 samples) acts as a *held-out exam* the model has never seen.

`stratify=y` ensures each class is proportionally represented in both sets — 
essential for maintaining the dataset's balanced nature.  
`shuffle=True` removes any ordering bias (the raw CSV lists all Setosa first, then Versicolor, then Virginica).


In [ ]:
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, shuffle=True, stratify=y
)

# Fit scaler ONLY on training data, then transform both sets
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f"Training set : {X_train.shape[0]} samples")
print(f"Test set     : {X_test.shape[0]} samples")
print(f"\nClass distribution (train): {dict(y_train.value_counts().sort_index())}")
print(f"Class distribution (test) : {dict(y_test.value_counts().sort_index())}")


## 5. Hyperparameter Tuning — Finding the Optimal K

K controls the bias-variance tradeoff:
- **K=1** → pure memorisation, overfits noise in training data
- **K=100** → consults too many neighbours, loses local structure (underfitting)
- **The Elbow** → the K where error rate bottoms out before climbing again

We test odd values only (1, 3, 5, … 29) to avoid tie-breaking ambiguity in voting.


In [ ]:
k_values = list(range(1, 30, 2))  # odd K values: 1, 3, 5 … 29
error_rates = []

for k in k_values:
    knn_temp = KNeighborsClassifier(n_neighbors=k)
    knn_temp.fit(X_train, y_train)
    preds_temp = knn_temp.predict(X_test)
    error_rates.append(1 - accuracy_score(y_test, preds_temp))

# Identify the elbow — lowest error rate
optimal_k = k_values[error_rates.index(min(error_rates))]
min_error = min(error_rates)

# Plot
fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(k_values, error_rates, marker='o', color='#2C3E50', linewidth=2,
        markersize=8, markerfacecolor='white', markeredgewidth=2)
ax.scatter([optimal_k], [min_error], color='#E74C3C', s=180, zorder=5,
           label=f'Optimal K = {optimal_k}  (error = {min_error:.3f})')
ax.axvline(x=optimal_k, color='#E74C3C', linestyle='--', alpha=0.6)
ax.set_title('Error Rate vs K — The Elbow Curve', fontsize=13, fontweight='bold')
ax.set_xlabel('K (Number of Neighbours)')
ax.set_ylabel('Error Rate on Test Set')
ax.set_xticks(k_values)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('elbow_curve.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"Selected K = {optimal_k}  |  Test Error = {min_error:.4f}  |  Accuracy = {1 - min_error:.4f}")


## 6. Training the Final KNN Model

With K selected from the elbow analysis, we train the definitive model.


In [ ]:
knn_model = KNeighborsClassifier(n_neighbors=optimal_k)
knn_model.fit(X_train, y_train)
y_pred = knn_model.predict(X_test)

print(f"KNN classifier trained  |  K = {optimal_k}")
print(f"Test Accuracy           : {accuracy_score(y_test, y_pred):.4f}")


## 7. Evaluation — Confusion Matrix & F1 Score

**Why accuracy alone isn't enough:**  
A model that always predicts the majority class on a 95/5 imbalanced dataset would 
score 95% accuracy while being completely useless. We need richer diagnostics.

**Confusion Matrix** shows exactly *which* classes are being confused with which.  
**Precision** = "of all predictions for class X, how many were actually X?" (trustworthiness)  
**Recall** = "of all real class X samples, how many did we catch?" (sensitivity)  
**F1 Score** = harmonic mean of precision and recall — penalises extreme imbalance between the two.


In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[0], linewidths=0.5, linecolor='white', annot_kws={'size': 13})
axes[0].set_title('Confusion Matrix (Counts)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')

# Normalised %
sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Greens',
            xticklabels=target_names, yticklabels=target_names,
            ax=axes[1], linewidths=0.5, linecolor='white', annot_kws={'size': 13})
axes[1].set_title('Confusion Matrix (Normalised)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Actual Label')
axes[1].set_xlabel('Predicted Label')

plt.suptitle(f'KNN Classifier (K={optimal_k}) — Performance Diagnosis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names))
f1_macro    = f1_score(y_test, y_pred, average='macro')
f1_weighted = f1_score(y_test, y_pred, average='weighted')
print(f"F1 Score — Macro    : {f1_macro:.4f}")
print(f"F1 Score — Weighted : {f1_weighted:.4f}")


## 8. Algorithm Comparison (Differentiation)

KNN is an excellent baseline, but how does it compare to other classifiers on this data?  
We benchmark against **Decision Tree** (rule-based splits) and **Logistic Regression** 
(linear probabilistic model) under identical preprocessing conditions.


In [ ]:
contenders = {
    f'KNN  (K={optimal_k})': KNeighborsClassifier(n_neighbors=optimal_k),
    'Decision Tree    ': DecisionTreeClassifier(random_state=RANDOM_SEED, max_depth=4),
    'Logistic Regression': LogisticRegression(random_state=RANDOM_SEED, max_iter=300, C=1.0)
}

records = []
for name, model in contenders.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    records.append({
        'Model'       : name.strip(),
        'Accuracy'    : round(accuracy_score(y_test, preds), 4),
        'F1 (Macro)'  : round(f1_score(y_test, preds, average='macro'), 4),
        'F1 (Weighted)': round(f1_score(y_test, preds, average='weighted'), 4)
    })

results_df = pd.DataFrame(records).set_index('Model')
print(results_df.to_string())

# Bar chart comparison
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(results_df))
w = 0.26
palette = ['#3498DB', '#E74C3C', '#2ECC71']
metrics = ['Accuracy', 'F1 (Macro)', 'F1 (Weighted)']

for i, (metric, color) in enumerate(zip(metrics, palette)):
    bars = ax.bar(x + (i - 1) * w, results_df[metric], w,
                  label=metric, color=color, alpha=0.85, edgecolor='white')
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.003,
                f'{h:.3f}', ha='center', va='bottom', fontsize=8, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(results_df.index, fontsize=10)
ax.set_ylim(0.88, 1.06)
ax.set_title('Algorithm Comparison — Accuracy & F1 Scores', fontsize=13, fontweight='bold')
ax.set_ylabel('Score')
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig('algorithm_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


## 9. Summary & Key Takeaways

### Results at a Glance
| Metric | Value |
|--------|-------|
| Algorithm | K-Nearest Neighbors |
| Optimal K | *(see elbow curve)* |
| Test Accuracy | *(see model output above)* |
| F1 Score (Macro) | *(see classification report)* |

### What we learned

**StandardScaler is non-negotiable for KNN.**  
Raw Iris features span different value ranges. Without scaling, sepal length would 
dominate distance calculations purely because its absolute values are larger — not 
because it's more informative. Post-scaling, all four features contribute proportionally.

**The elbow method is principled, not arbitrary.**  
We didn't just pick K=5 because it's "common." We measured actual error rates across 
15 different K values and identified where increasing K stops improving and starts hurting. 
That's rigorous hyperparameter selection.

**F1 Score > Accuracy as your headline metric.**  
On balanced datasets like Iris, both tell the same story — but F1 catches the nuance of 
*which* classes the model struggles with. The normalised confusion matrix shows percentage 
of each class correctly identified, which per-class accuracy hides.

**KNN is competitive here for good reason.**  
The Iris decision boundary is relatively smooth and local — exactly the conditions under which 
KNN thrives. Algorithms that assume linearity (Logistic Regression) may struggle at the 
Versicolor/Virginica boundary where the classes partially overlap.

---
*Project 2 — DecodeLabs AI Internship | Supervised Learning Pipeline*
